<a href="https://colab.research.google.com/github/mSharifHub/DAFN/blob/main/biliearn_interpolation_testing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import time
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
SAVE_DIR = "/content/drive/MyDrive/DAFN_Project"
os.makedirs(SAVE_DIR, exist_ok=True)

In [ ]:
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Weights & graphs will be saved to: {SAVE_DIR}")

Using device: cuda
GPU: NVIDIA L4
Weights & graphs will be saved to: /content/drive/MyDrive/DAFN_Project


In [ ]:
class TranslatedMNIST(Dataset):

  def __init__(self, root="./data", train=True, canvas_size=64,
                 fixed_positions=False, seed=42, add_clutter=False):
        self.mnist = datasets.MNIST(
            root=root,
            train=train,
            download=True,
            transform=transforms.ToTensor()
        )

        self.canvas_size = canvas_size
        self.digit_size = 28
        self.max_offset = canvas_size - self.digit_size
        self.fixed_positions = fixed_positions
        self.add_clutter = add_clutter

        # Pre-generate deterministic coordinates for validation/test reproducibility
        if fixed_positions:
            g = torch.Generator().manual_seed(seed)
            self.offsets = torch.randint(
                0, self.max_offset + 1, (len(self.mnist), 2), generator=g
            )


  def __len__(self):
        return len(self.mnist)

  def __getitem__(self, idx):
        digit, label = self.mnist[idx]
        canvas = torch.zeros(1, self.canvas_size, self.canvas_size)


        if self.fixed_positions:
            top, left = self.offsets[idx].tolist()
        else:
            top = torch.randint(0, self.max_offset + 1, (1,)).item()
            left = torch.randint(0, self.max_offset + 1, (1,)).item()

        canvas[:, top:top + self.digit_size, left:left + self.digit_size] = digit


        if self.add_clutter:
            for _ in range(4):
                r_idx = torch.randint(0, len(self.mnist), (1,)).item()
                r_digit, _ = self.mnist[r_idx]
                ry = torch.randint(0, 22, (1,)).item()
                rx = torch.randint(0, 22, (1,)).item()
                cy = torch.randint(0, self.canvas_size - 6, (1,)).item()
                cx = torch.randint(0, self.canvas_size - 6, (1,)).item()
                canvas[:, cy:cy+6, cx:cx+6] = torch.maximum(
                    canvas[:, cy:cy+6, cx:cx+6], r_digit[:, ry:ry+6, rx:rx+6] * 0.6
                )

        return canvas, label




In [ ]:
# Split train into 54,000 train + 6,000 validation, plus 10,000 test
full_train = TranslatedMNIST(train=True, canvas_size=64, fixed_positions=False)
train_subset, val_subset = torch.utils.data.random_split(
    full_train, [54000, 6000], generator=torch.Generator().manual_seed(SEED)
)
test_dataset = TranslatedMNIST(train=False, canvas_size=64, fixed_positions=True, seed=99)

BATCH_SIZE = 128
train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_subset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train: {len(train_subset)} | Val: {len(val_subset)} | Test: {len(test_dataset)}")

Train: 54000 | Val: 6000 | Test: 10000


In [10]:
class DifferentiableActiveFoveationNet(nn.Module):
    def __init__(self, num_classes=10, num_glimpses=4, patch_size=20,
                 hidden_dim=128, min_scale=0.25, max_scale=0.70):
        super().__init__()
        self.num_glimpses = num_glimpses
        self.patch_size = patch_size
        self.hidden_dim = hidden_dim
        self.min_scale = min_scale
        self.max_scale = max_scale

        # 1. Peripheral Vision Encoder (processes blurry 16x16 global context)
        self.peripheral_encoder = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 16x16 -> 8x8
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 8x8 -> 4x4
            nn.Flatten(),
            nn.Linear(32 * 4 * 4, 128),
            nn.ReLU(inplace=True),
        )

        # 2. Learned Initial Hidden Memory State (h_0)
        self.h0 = nn.Parameter(torch.zeros(1, hidden_dim))

        # 3. Deterministic Spatial Planner: [peripheral_feat, h_{t-1}] -> (tx, ty, scale)
        self.planner = nn.Sequential(
            nn.Linear(128 + hidden_dim, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, 3)
        )
        # Initialize planner to start near center with a wider scale (~0.58)
        # to avoid flat black background zero-gradient local minima early on
        nn.init.normal_(self.planner[-1].weight, mean=0.0, std=0.01)
        self.planner[-1].bias.data = torch.tensor([0.0, 0.0, 1.0])

        # 4. High-Resolution Foveal Patch Encoder (processes 20x20 bilinear crop)
        self.foveal_encoder = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 20x20 -> 10x10
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 10x10 -> 5x5
            nn.Flatten(),
            nn.Linear(64 * 5 * 5, 128),
            nn.ReLU(inplace=True),
        )

        # 5. Gaze Coordinate Embedding + Recurrent Memory (enables BPTT)
        self.loc_encoder = nn.Sequential(
            nn.Linear(3, 32),
            nn.ReLU(inplace=True),
        )
        self.memory_rnn = nn.GRUCell(input_size=128 + 32, hidden_size=hidden_dim)

        # 6. Visual Classifier (reads ONLY foveal memory h_t)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.ReLU(inplace=True),
            nn.Linear(64, num_classes)
        )

    def sample_fovea(self, high_res_img, tx, ty, s):
        """Differentiable Spatial Transformer bilinear sampler (Geometric Compass)."""
        B, C, _, _ = high_res_img.shape
        zeros = torch.zeros_like(s)

        # Affine matrix: [[s, 0, tx], [0, s, ty]]
        row1 = torch.stack([s, zeros, tx], dim=1)
        row2 = torch.stack([zeros, s, ty], dim=1)
        theta = torch.stack([row1, row2], dim=1)  # (B, 2, 3)

        grid = F.affine_grid(
            theta, size=(B, C, self.patch_size, self.patch_size), align_corners=False
        )
        patch = F.grid_sample(
            high_res_img, grid, mode="bilinear", padding_mode="zeros", align_corners=False
        )
        return patch

    def forward(self, x, return_trace=False):
        B = x.size(0)

        # Step A: Extract cheap blurry peripheral vision (64x64 -> 16x16)
        peripheral_img = F.avg_pool2d(x, kernel_size=4, stride=4)
        periph_feat = self.peripheral_encoder(peripheral_img)

        # Step B: Initialize memory state h_0
        h = self.h0.expand(B, -1).contiguous()

        step_logits = []
        gaze_coords = []
        foveal_patches = []

        # Step C: Sequential active foveation loop (unrolled for BPTT)
        for _ in range(self.num_glimpses):
            planner_in = torch.cat([periph_feat, h], dim=1)
            raw_u = self.planner(planner_in)

            # Constrain normalized gaze coordinates and continuous scale
            tx = torch.tanh(raw_u[:, 0]) * 0.85
            ty = torch.tanh(raw_u[:, 1]) * 0.85
            s = self.min_scale + (self.max_scale - self.min_scale) * torch.sigmoid(raw_u[:, 2])

            # Differentiable sub-pixel extraction
            patch = self.sample_fovea(x, tx, ty, s)

            # Encode foveal visual features + where the model looked
            fovea_feat = self.foveal_encoder(patch)
            gaze_vec = torch.stack([tx, ty, s], dim=1)
            loc_feat = self.loc_encoder(gaze_vec)

            # Update internal hidden memory state h_t
            h = self.memory_rnn(torch.cat([fovea_feat, loc_feat], dim=1), h)

            # Per-step classification logits
            logits_t = self.classifier(h)
            step_logits.append(logits_t)

            if return_trace:
                gaze_coords.append(gaze_vec.detach().cpu())
                foveal_patches.append(patch.detach().cpu())

        final_logits = step_logits[-1]

        if return_trace:
            return final_logits, {
                "peripheral_img": peripheral_img.detach().cpu(),
                "gaze_coords": torch.stack(gaze_coords, dim=1),       # (B, T, 3)
                "foveal_patches": torch.stack(foveal_patches, dim=1), # (B, T, 1, P, P)
                "step_logits": torch.stack(step_logits, dim=1).detach().cpu()
            }
        return final_logits, step_logits


# Quick sanity check on shapes and parameter count
MODEL_CONFIG = {
    "num_classes": 10,
    "num_glimpses": 4,
    "patch_size": 20,
    "hidden_dim": 128,
    "min_scale": 0.25,
    "max_scale": 0.70,
}

model = DifferentiableActiveFoveationNet(**MODEL_CONFIG).to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model initialized | Trainable parameters: {total_params:,}")

Model initialized | Trainable parameters: 448,013
